# Implicit Neural Representations for 1D Signal Reconstruction

This notebook demonstrates an **Implicit Neural Representation (INR)** applied to a 1D signal.
An INR is a small MLP that learns to map continuous coordinates → signal values,
acting as a compact, differentiable function approximator.

**What we'll do:**
1. Generate a noisy composite sinusoidal signal
2. Encode time coordinates with Fourier features
3. Train an MLP on **20% of the samples** (sparse supervision)
4. Evaluate denoising quality on all 1000 points
5. Test interpolation by querying at a denser grid (1200 points) the model was never trained on

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

## 1. Signal Generation

We use a composite sinusoidal signal:

$$y(t) = 2\sin(t) + 1.5\cos(2t) + 0.5\sin(3t)$$

The FFT confirms three dominant frequency components at $f \approx 0.16$, $0.32$, and $0.48$ Hz.

In [ ]:
t = np.linspace(0, 10, 1000)
y = 2 * np.sin(t) + 1.5 * np.cos(2 * t) + 0.5 * np.sin(3 * t)

fft_val = np.fft.fft(y)
frequencies = np.fft.fftfreq(len(y), d=(t[1] - t[0]))
positive_freqs = frequencies[frequencies > 0]
magnitude = np.abs(fft_val[frequencies > 0])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, y, color='steelblue')
axes[0].set_title("Clean Signal")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

axes[1].plot(positive_freqs, magnitude, color='darkorange')
axes[1].set_title("Frequency Spectrum (FFT)")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Magnitude")
axes[1].set_xlim(0, 5)

plt.tight_layout()
plt.show()

## 2. Adding Noise

We corrupt the signal with zero-mean Gaussian noise ($\sigma = 0.05$)
and measure the Signal-to-Noise Ratio (SNR) to quantify degradation.

In [ ]:
noise = np.random.normal(0, 0.05, size=t.shape)
y_noisy = y + noise

signal_power = np.mean(y ** 2)
noise_power = np.mean(noise ** 2)
snr = 10 * np.log10(signal_power / noise_power)
print(f"SNR: {snr:.2f} dB")

plt.figure(figsize=(10, 4))
plt.plot(t, y, label="Clean Signal", linewidth=1.5)
plt.plot(t, y_noisy, label="Noisy Signal", alpha=0.55)
plt.title(f"Signal with Gaussian Noise  (SNR = {snr:.1f} dB)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Fourier Feature Positional Encoding

Raw coordinates fed to a plain MLP suffer from *spectral bias* — the network
preferentially learns low-frequency components and struggles with fine detail.
Fourier feature encoding (Tancik et al., 2020) maps each coordinate $x \in [0,1]$ to

$$\gamma(x) = \bigl[\sin(2^0 \pi x),\; \cos(2^0 \pi x),\; \ldots,\; \sin(2^{L-1} \pi x),\; \cos(2^{L-1} \pi x)\bigr]$$

With $L = 5$ frequencies, each scalar coordinate becomes a **10-dimensional** vector.

In [ ]:
def positional_encoding(x, num_freqs=5):
    pe = []
    for i in range(num_freqs):
        freq = 2 ** i
        pe.append(np.sin(freq * np.pi * x))
        pe.append(np.cos(freq * np.pi * x))
    return np.stack(pe, axis=-1)

t_norm = t / t.max()          # normalize to [0, 1]
enc_t = positional_encoding(t_norm, num_freqs=5)
print(f"Encoding shape: {enc_t.shape}")  # (1000, 10)

## 4. INR Architecture

A compact 3-layer MLP: `10 → 128 → 128 → 1`  
Input: Fourier-encoded time coordinate  
Output: predicted signal amplitude at that coordinate

In [ ]:
class INR(nn.Module):
    def __init__(self, encoding_dim=10, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(encoding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = INR(encoding_dim=10, hidden_dim=128).to(device)
enc_t_tensor = torch.tensor(enc_t, dtype=torch.float32).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device     : {device}")
print(f"Parameters : {total_params:,}")

## 5. Training

We train on only **20% of the time-steps** (200 out of 1000), chosen at random.
This simulates sparse, irregular sampling — a common real-world scenario.
The INR learns the underlying continuous function and can predict anywhere,
not just at training locations.

In [ ]:
def train_model(model, enc_t_tensor, y_noisy, epochs=5000, lr=1e-3, sample_frac=0.2):
    y_tensor = torch.tensor(y_noisy, dtype=torch.float32).to(device)

    n = len(enc_t_tensor)
    idx = np.random.choice(n, size=int(sample_frac * n), replace=False)
    x_train = enc_t_tensor[idx]
    y_train = y_tensor[idx]

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    losses = []
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = model(x_train).squeeze(-1)
        loss = loss_fn(pred, y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if epoch % 1000 == 0:
            print(f"Epoch {epoch:5d}  loss = {loss.item():.6f}")

    return losses


losses = train_model(model, enc_t_tensor, y_noisy, epochs=5000)

plt.figure(figsize=(8, 3))
plt.semilogy(losses, linewidth=0.8)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (log scale)")
plt.tight_layout()
plt.show()

## 6. Evaluation

We compare the INR reconstruction against the noisy baseline on the full 1000-point grid.
Both are measured against the ground-truth clean signal.

In [ ]:
model.eval()
with torch.no_grad():
    recon = model(enc_t_tensor).squeeze(-1).cpu().numpy()

def print_metrics(ref, pred, label):
    mse = np.mean((ref - pred) ** 2)
    mae = np.mean(np.abs(ref - pred))
    r   = np.corrcoef(ref, pred)[0, 1]
    print(f"{label:35s}  MSE={mse:.6f}  MAE={mae:.6f}  r={r:.4f}")

print_metrics(y, y_noisy, "Noisy  vs Clean (baseline)")
print_metrics(y, recon,   "INR Reconstruction vs Clean")

plt.figure(figsize=(10, 4))
plt.plot(t, y_noisy, label="Noisy Signal",       alpha=0.45)
plt.plot(t, y,       label="Clean Signal",        linewidth=1.5)
plt.plot(t, recon,   label="INR Reconstruction", linewidth=1.5, linestyle='--')
plt.title("INR Signal Reconstruction")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Interpolation Test

Because the INR represents a **continuous function**, we can query it at any coordinate —
including ones never seen during training.
Here we evaluate at a denser grid of **1200 points** (vs. 1000 training-range points)
to confirm the network generalises smoothly.

In [ ]:
t_dense = np.linspace(0, 10, 1200)
y_dense = 2 * np.sin(t_dense) + 1.5 * np.cos(2 * t_dense) + 0.5 * np.sin(3 * t_dense)

enc_dense = positional_encoding(t_dense / t_dense.max(), num_freqs=5)
enc_dense_tensor = torch.tensor(enc_dense, dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    out_dense = model(enc_dense_tensor).squeeze(-1).cpu().numpy()

plt.figure(figsize=(10, 4))
plt.plot(t_dense, y_dense,   label="Ground Truth (1200 pts)", linewidth=1.5)
plt.plot(t_dense, out_dense, label="INR Interpolation",       linewidth=1.5, linestyle='--')
plt.title("INR Interpolation to Denser Grid")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.tight_layout()
plt.show()

print_metrics(y_dense, out_dense, "INR Interpolation vs Ground Truth")